# Notebook 01: Bienvenida y Tu Primer Data Doc

## ¡Bienvenido al Taller de Great Expectations!

En los próximos módulos aprenderás a:
- Validar la calidad de tus datos
- Documentar reglas de negocio
- Generar reportes profesionales
- Automatizar validaciones en producción

**Duración**: 20 minutos
**Nivel**: Principiante

## ¿Qué es Calidad de Datos?

La **calidad de datos** es el grado en que los datos cumplen con los requisitos de uso previstos.

### ¿Por qué importa?

- **Decisiones incorrectas**: Datos malos → Decisiones malas
- **Costos**: Se estima que cuesta 15-25% de los ingresos
- **Riesgos**: Incumplimiento regulatorio, pérdida de confianza

### Ejemplo Real

Imagina un e-commerce donde:
-  Precios negativos → Pérdidas financieras
-  IDs de cliente nulos → No puedes contactar al cliente
-  Fechas futuras → Reportes incorrectos

## ¿Qué es Great Expectations?

**Great Expectations (GX)** es una herramienta open-source para:

1. **Validar** datos contra reglas definidas
2. **Documentar** esas reglas automáticamente
3. **Generar reportes** visuales (Data Docs)
4. **Automatizar** validaciones en pipelines

### Filosofía

> "Las expectativas sobre tus datos deben ser explícitas, versionadas y testeables"

In [ ]:
# Importar librerías
import great_expectations as gx
import pandas as pd

print(f"Great Expectations versión: {gx.__version__}")

## Tu Primera Validación

Vamos a validar un dataset de ventas con problemas de calidad.

In [ ]:
# Cargar datos
df = pd.read_csv("../data/ventas_sucias.csv")

print(f"Total de registros: {len(df)}")
print(f"\nPrimeras filas:")
df.head()

In [ ]:
# Análisis rápido de problemas
print("=== PROBLEMAS DETECTADOS ===")
print(f"\nValores nulos:")
print(df.isnull().sum())

print(f"\nPrecios negativos: {(df['price'] < 0).sum()}")
print(f"Cantidades <= 0: {(df['quantity'] <= 0).sum()}")

## Crear Tu Primera Expectativa

Una **Expectation** es una regla sobre tus datos. Por ejemplo:
- "Los precios deben ser positivos"
- "El customer_id no debe ser nulo"

In [ ]:
# Inicializar contexto
context = gx.get_context(mode="ephemeral")

# Configurar datasource
datasource = context.data_sources.add_pandas(name="ventas_ds")
asset = datasource.add_dataframe_asset(name="ventas")
batch_def = asset.add_batch_definition_whole_dataframe("batch_completo")

print(" Contexto configurado")

In [ ]:
# Crear suite de expectativas
suite = context.suites.add(gx.ExpectationSuite(name="mi_primera_suite"))

# Agregar expectativas simples
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToNotBeNull(column="customer_id")
)

suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column="price",
        min_value=0.01
    )
)

suite.save()
print(f" Suite creada con {len(suite.expectations)} expectativas")

## Ejecutar la Validación

In [ ]:
# Crear validation definition
validation_def = context.validation_definitions.add(
    gx.ValidationDefinition(
        data=batch_def,
        suite=suite,
        name="primera_validacion"
    )
)

# Ejecutar validación
resultado = validation_def.run(batch_parameters={"dataframe": df})

print("\n" + "="*50)
print("RESULTADO DE LA VALIDACIÓN")
print("="*50)
print(f"\n¿Validación exitosa?: {resultado.success}")
print(f"Expectativas evaluadas: {len(resultado.results)}")
print(f"Expectativas que pasaron: {sum(1 for r in resultado.results if r.success)}")
print(f"Expectativas que fallaron: {sum(1 for r in resultado.results if not r.success)}")

## Tu Primer Data Doc

Ahora viene la magia: Great Expectations genera automáticamente documentación HTML interactiva.

In [ ]:
# Generar Data Docs
context.build_data_docs()

print("\n" + "="*50)
print(" Data Docs generados exitosamente!")
print("="*50)
print("\nAbriendo en tu navegador...")

# Abrir en navegador
context.open_data_docs()

## ¿Qué Ves en el Data Doc?

En la página que se abrió, observa:

1. **Overview**: Resumen de la validación
2. **Expectation Suite**: Todas tus reglas documentadas
3. **Validation Results**: Resultados detallados
4. **Statistics**: Gráficos y métricas

### Ventajas

-  **Visual**: Fácil de entender para no-técnicos
-  **Compartible**: Puedes enviar el link
-  **Automático**: Se genera sin esfuerzo
-  **Profesional**: Listo para stakeholders

##  Resumen del Notebook

En 20 minutos has aprendido:

1.  Qué es calidad de datos y por qué importa
2.  Qué es Great Expectations
3.  Cómo crear tu primera expectativa
4.  Cómo ejecutar una validación
5.  Cómo generar tu primer Data Doc

